# Capstone — Content decline review prioritization

This notebook documents the evidence behind the deployed research page. The reported Week 5–7 metrics are transcribed from the saved notebooks and recorded in `work/outputs/capstone_metrics.json`. This project uses the bundled 30,000-row anonymized starter sample; it does **not** claim to have analyzed the separate 78.8M-row warehouse.

**Research page:** `docs/index.html` (after GitHub Pages is enabled).

## 1. Question

**Can observable content and performance signals rank pages labelled as declining more effectively than a simple rule on a client-held-out test set?**

The unit is one content item. The intended decision is which pages deserve earlier human investigation. The output is a ranked review queue, reason codes, and suggested human review path—not an automatic edit instruction.

A false positive can waste editorial time; a false negative can leave a potential issue unreviewed. The model supports triage, not autonomous publishing.

In [ ]:
from pathlib import Path
import json
import pandas as pd

REPO = Path.cwd()
if not (REPO / "work/outputs/capstone_metrics.json").exists():
    REPO = Path("/content/Flyrank-ML-Internship")

metrics_path = REPO / "work/outputs/capstone_metrics.json"
metrics = json.loads(metrics_path.read_text())
print(metrics["paper_title"])
print("Scope:", metrics["analysis_scope"])

## 2. Data

The experiments use `data/raw/content_refresh_anonymized.csv`: 30,000 rows × 44 source columns, one row per content item, 32 pseudonymized clients. The preparation script creates a 52-column feature vector. Metrics are trailing-90-day aggregates ending at export time; trend fields compare the latest 30 days with the preceding 30 days. The exact calendar boundaries of this starter snapshot are not specified.

The internship separately documents a gated warehouse build v20260703 with 78,835,655 daily performance rows from 2025-01-27 to 2026-06-30. Those warehouse rows are **not** the source of the reported metrics.

Excluded from features: `trend_direction`, `trend_pct`, and `is_declining_label` (target construction); `content_id` and `client_id` (identifiers for joins/grouping only). No client names, domains, URLs, page titles, or raw queries are published. Missingness varies by content type; rates use a 0–100 scale; `avg_position=0` means no position data.

In [ ]:
raw_path = REPO / "data/raw/content_refresh_anonymized.csv"
raw = pd.read_csv(raw_path)
print("Raw shape:", raw.shape)
print("Unique pseudonymized clients:", raw["client_id"].nunique())
print("Trend-direction counts:")
display(raw["trend_direction"].value_counts(dropna=False).to_frame("rows"))

assert len(raw) == 30000
assert raw["client_id"].nunique() == 32

## 3. Methodology

**Label:** `is_declining_label = 1` when `trend_direction == "down"`, else 0. In the starter sample, down means impressions fell by more than 20% from the preceding 30-day period. New, flat, up, and stable are grouped into class 0. This is a historical proxy, not a future outcome.

**Features:** the repository's official 18 numeric and 8 categorical variables (26 total), covering content/keyword attributes, age/freshness, and trailing-90-day search/engagement aggregates. Logistic Regression uses numeric median imputation with missingness indicators and scaling, categorical mode imputation and one-hot encoding, balanced class weights, and seed 42.

**Baseline:** for the Week 5 same-split comparison, one point for `content_age_days ≥ 180` and one point for `impressions_90d ≥ 500`. This simple rule differs from the richer weighted baseline score used by the Week 7 operational queue.

**Validation:** GroupShuffleSplit 80/20 by `client_id`, seed 42. Saved run: 23,837 training rows / 25 clients; 6,163 test rows / 7 clients; zero client overlap. Test decline-label rate: 51.1%. Week 6 adds five-fold GroupKFold out-of-fold evaluation.

**Leakage audit:** direct target fields and IDs are excluded and client overlap is zero. However, 90-day performance features may overlap the label window. The repository does not prove predictor timestamps precede the label period; prospective temporal leakage is unresolved. A future test must build features strictly before the label window and evaluate on a later sealed period.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import sys
sys.path.insert(0, str(REPO / "scripts"))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

prepared = pd.read_csv(REPO / "data/processed/refresh_feature_vector.csv") if (REPO / "data/processed/refresh_feature_vector.csv").exists() else None
if prepared is None:
    import subprocess
    subprocess.run([sys.executable, str(REPO / "scripts/01_prepare_features.py")], cwd=REPO, check=True)
    prepared = pd.read_csv(REPO / "data/processed/refresh_feature_vector.csv")

features = list(MODEL_NUMERIC_FEATURES) + list(MODEL_CATEGORICAL_FEATURES)
forbidden = {"content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"}
assert not forbidden.intersection(features)

frame = prepared.dropna(subset=["is_declining_label", "client_id"]).copy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(frame, frame["is_declining_label"], groups=frame["client_id"]))
train_clients = set(frame.iloc[train_idx]["client_id"])
test_clients = set(frame.iloc[test_idx]["client_id"])
assert train_clients.isdisjoint(test_clients)
print("Feature count:", len(features))
print("Train/test rows:", len(train_idx), len(test_idx))
print("Train/test clients:", len(train_clients), len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

## 4. Results (vs baseline)

Saved Week 5 results on the same 6,163 held-out rows:

| Method | Precision@20 | Precision@50 | Average Precision |
|---|---:|---:|---:|
| Week 4 rule baseline | 0.40 | 0.52 | 0.4980 |
| Logistic Regression | **0.70** | **0.72** | **0.6044** |

The held-out declining-label base rate is 51.1%. Week 6's separate five-fold GroupKFold out-of-fold results are AP 0.6678, Precision@20 0.75, Precision@50 0.74. This is a different evaluation construction, not evidence of model improvement.

Week 5 top permutation-importance features: log_impressions_90d 0.0734; log_clicks_90d 0.0585; avg_position 0.0372; content_age_days 0.0310; impression_tier 0.0295; log_sessions_90d 0.0191; word_count 0.0101; scroll_rate 0.0093.

Week 7's separate full-sample queue uses Random Forest predictions and a 70% model probability + 30% normalized baseline score. Its action counts are summarized separately; they are not validation metrics.

In [ ]:
comparison = pd.DataFrame(metrics["week5_same_holdout"])
display(comparison)
print("Test-set decline-label rate:", metrics["validation"]["test_decline_rate"])
print("Week 6 GroupKFold OOF:", metrics["week6_groupkfold_oof"])

## 5. Limitations

- Results use the 30,000-row starter sample, not the full 78.8M-row warehouse.
- The target is historical impression decline, not Google ranking movement, business loss, or future refresh success.
- Temporal availability is unresolved: trailing-90-day predictors may overlap the label period.
- Observational data does not establish that refreshing causes recovery.
- The single grouped holdout contains seven held-out clients; performance may vary by client and time.
- Seasonality, demand, tracking differences, cannibalization, missingness, and editorial selection may influence results.
- Queue confidence is an operational threshold label, not calibrated probability of refresh success.

The defensible claim is narrow: on this sample and saved grouped evaluation, Logistic Regression ranked historical decline labels better than the simple rule at the reported cutoffs. This supports human review prioritization only.

In [ ]:
assert metrics["data"]["rows"] == 30000
assert metrics["data"]["client_groups"] == 32
assert metrics["validation"]["client_overlap"] == 0
assert "Temporal overlap" in " ".join(metrics["caveats"])
print("Scope and caveat checks passed.")

## 6. Ranked recommendations

The Week 7 queue is a separate operational ranking built from the saved best-model predictions (Random Forest) and a 70/30 model/baseline score blend.

| Priority | Review signal | Human action |
|---:|---|---|
| 1 | Declining content with demand | Diagnose intent, seasonality, cannibalization, and factual freshness before edits |
| 2 | CTR opportunity | Review search intent and SERP context before title/snippet tests |
| 3 | Engagement opportunity | Check usefulness, structure, page experience, and tracking |
| 4 | Thin visible content | Expand only where user needs are missing and evidence supports it |
| 5 | Stale visible / page-one decay risk | Verify material changes; age alone is not a reason to refresh |
| 6 | General monitoring | Monitor; do not force an edit |

Saved Week 7 suggested-action counts: monitor 13,083; refresh 8,188; refresh + CTR review 6,654; refresh + engagement review 1,993; expand + refresh 82.

Human approval is mandatory. Never automate publishing, rewriting, deletion, redirects, canonical/schema/metadata changes, or sensitive editorial decisions. A high score is a queue position, not an instruction.

In [ ]:
print("Week 7 queue summary:")
display(pd.Series(metrics["week7_queue"]["suggested_actions"], name="rows").to_frame())
print("Score formula:", metrics["week7_queue"]["score_formula"])

## 7. Artifacts the paper embeds

The deployed paper embeds three SVG figures:
- `docs/assets/holdout-comparison.svg` — same-split model vs baseline metrics.
- `docs/assets/permutation-importance.svg` — held-out feature permutation importance.
- `docs/assets/action-mix.svg` — Week 7 suggested-action counts.

The matching source figures are under `work/figures/`; the metrics receipt is `work/outputs/capstone_metrics.json`.

In [ ]:
required_assets = [
    REPO / "docs/assets/holdout-comparison.svg",
    REPO / "docs/assets/permutation-importance.svg",
    REPO / "docs/assets/action-mix.svg",
    REPO / "work/outputs/capstone_metrics.json",
]
for asset in required_assets:
    print(asset.relative_to(REPO), "OK" if asset.exists() else "MISSING")

## Self-check

- [x] All seven notebook sections contain evidence-based narrative and code checks.
- [ ] Run this completed capstone notebook top-to-bottom in the project environment.
- [x] The paper distinguishes the 30,000-row starter sample from the 78.8M-row warehouse.
- [x] Same-split model vs baseline metrics and the 51.1% test base rate are shown.
- [x] Temporal leakage risk is disclosed, not hidden.
- [x] The paper includes all nine required sections, including the five-sentence abstract and FlyRank data credit.
- [ ] Confirm GitHub Pages deployment and replace `submission/paper_url.txt` with the exact live URL.
- [x] ML-12 demo outline, social-post cut, and employer summary are included below.

## ML-12 closing material

**Five-minute demo outline**
1. 0:00–0:45 — Frame the decision: prioritize content for human review.
2. 0:45–1:30 — Explain the anonymized sample, target, and leakage exclusions.
3. 1:30–2:30 — Show the client-grouped holdout and same-split baseline comparison.
4. 2:30–3:30 — Explain Week 7 reason codes, ranked queue, and human-review policy.
5. 3:30–4:30 — Discuss temporal leakage risk and limitations.
6. 4:30–5:00 — Close with reproducibility and data credit.

**Social-post cut**

I built a content-decline review-prioritization prototype on FlyRank’s anonymized 30k-row internship sample. On a client-grouped holdout, Logistic Regression achieved Precision@20 0.70 vs 0.40 for a simple age-and-impressions rule. The result is a human-review signal—not a causal claim about refresh impact or Google rankings.

**Employer-facing summary**

I developed an applied ML workflow to prioritize content pages for human review. I prepared an anonymized dataset, built a transparent rule baseline, trained and evaluated Logistic Regression using a client-grouped holdout, audited temporal leakage risks, and translated predictions into an explainable action queue. The work demonstrates reproducible model evaluation and responsible decision-support framing; it does not claim causal refresh impact or prospective ranking prediction.